# Round 2: refined within-cluster PCA + AoU projection

Round 1 clustered to the broad EUR superpop (`p=0.999`, deliberately loose) with a Mahalanobis ellipsoid fit on the whole-cohort PCA. Round 2 sharpens this to CEU/GBR specifically: recompute LD pruning and PCA using *only* CEU/GBR reference samples and *only* the AoU samples that passed round 1 -- LD structure and stratification within a single tight cluster differ from the whole-cohort or whole-superpop picture, so a PCA fit just within CEU/GBR resolves finer sub-structure round 1 couldn't see.

Reuses round 1's already-built, already-QC'd files (`OUT_PREFIX`, `PRUNE_PREFIX.prune.in`, the biallelic-filtered ACAF pgen, `agreeing_snps.ids`) rather than rebuilding from scratch -- round 2's pruning is restricted to start from round 1's already-pruned SNP set, so the result stays a subset of what's already extracted in ACAF (no need to re-touch the network-mounted ACAF source at all).

In [ ]:
import os

bin_dir = os.path.expanduser("~/bin")
if bin_dir not in os.environ["PATH"].split(":"):
    os.environ["PATH"] = f"{bin_dir}:{os.environ['PATH']}"

## Inputs

Everything under "from round 1" should already exist -- this notebook doesn't rebuild them.

In [ ]:
# os.path.expanduser resolves ~ here, in Python -- never pass a literal "~" into
# a bash variable via %%bash -s, since bash only expands ~ in literal command
# text, not in the *value* of a substituted variable ("$1", "$VAR", etc.)
WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
BUCKET_DIR = f"{WORKSPACE_BUCKET}/data/01_ancestry_filtering"   # matches submit_pca_r1.ipynb's BUCKET_DIR

# from round 1 (submit_pca_r1.ipynb)
OUT_PREFIX = f"{BUCKET_DIR}/1kg_all_hm3"            # QC'd, whole-cohort bfile
PRUNE_PREFIX = f"{BUCKET_DIR}/all_pruned"           # round 1's LD-pruned SNP list
PANEL_PATH = f"{BUCKET_DIR}/integrated_call_samples_v3.20130502.ALL.panel"
# both copied to the bucket by the post-hoc biallelic filter cell in submit_pca_r1.ipynb
ACAF_BIALLELIC_PREFIX = f"{BUCKET_DIR}/round1_pca/acaf_hm3_subset_biallelic"
AGREEING_SNPS_PATH = f"{BUCKET_DIR}/round1_pca/agreeing_snps.ids"

# from round 1 filtering (round1_filter.ipynb) -- confirm this matches what that
# notebook's OUT_DIR + prob_tag actually produced before running
ROUND1_KEEP_PATH = f"{BUCKET_DIR}/round1_pca/round1_keep_ids_eur_p999.txt"

# this round -- CEU/GBR specifically, tighter than round 1's full EUR superpop
CLUSTER = "ceu_gbr"
REF_POPS = ["CEU", "GBR"]

OUT_DIR = f"{BUCKET_DIR}/round2_pca"   # round 2 outputs written here
os.makedirs(OUT_DIR, exist_ok=True)

R2_PRUNE_PREFIX = os.path.join(OUT_DIR, f"r2_pruned_{CLUSTER}")
R2_PCA_PREFIX = os.path.join(OUT_DIR, f"r2_pca_{CLUSTER}")
R2_PROJECT_PREFIX = os.path.join(OUT_DIR, f"r2_projected_{CLUSTER}")
R2_REPROJECT_PREFIX = os.path.join(OUT_DIR, f"r2_pca_{CLUSTER}_reprojected")

print(BUCKET_DIR)
print(OUT_DIR)

## Reference sample list for this cluster

Panel population labels -> the 1000G sample IDs in `REF_POPS`, for `--keep`.

In [ ]:
import pandas as pd

panel = pd.read_csv(PANEL_PATH, sep="\t")
ref_keep_ids = panel.loc[panel["pop"].isin(REF_POPS), "sample"]

r2_ref_keep_path = os.path.join(OUT_DIR, f"r2_ref_keep_{CLUSTER}.ids")
ref_keep_ids.to_csv(r2_ref_keep_path, index=False, header=False)
print(f"{len(ref_keep_ids)} reference samples ({'+'.join(REF_POPS)}) -> {r2_ref_keep_path}")

## Round 2 LD pruning + PCA

Pruning starts from round 1's already-pruned SNP list (`--extract "${PRUNE_PREFIX}.prune.in"`), not the full QC'd variant set -- keeps `r2_pruned.prune.in` a subset of what ACAF already has extracted, so no re-extraction against the network-mounted ACAF source is needed for the projection step below.

`--nonfounders`: plink2 restricts frequency/PCA-type computations to founders by default. 1000G's "founder" designation is a pedigree artifact of the .fam file, not meaningful for population-structure PCA -- and with only ~190 CEU/GBR samples to begin with, dropping the ~5 non-founders further shrinks an already-small sample. Include everyone.

In [ ]:
%%bash -s "$OUT_PREFIX" "$PRUNE_PREFIX" "$r2_ref_keep_path" "$R2_PRUNE_PREFIX" "$R2_PCA_PREFIX"
set -e
OUT_PREFIX=$1
PRUNE_PREFIX=$2
REF_KEEP=$3
R2_PRUNE_PREFIX=$4
R2_PCA_PREFIX=$5

plink2 \
  --bfile "$OUT_PREFIX" \
  --keep "$REF_KEEP" \
  --extract "${PRUNE_PREFIX}.prune.in" \
  --nonfounders \
  --indep-pairwise 200kb 1 0.1 \
  --out "$R2_PRUNE_PREFIX"

wc -l "${R2_PRUNE_PREFIX}.prune.in"

plink2 \
  --bfile "$OUT_PREFIX" \
  --keep "$REF_KEEP" \
  --extract "${R2_PRUNE_PREFIX}.prune.in" \
  --nonfounders \
  --freq counts \
  --pca allele-wts 20 \
  --out "$R2_PCA_PREFIX"

ls -lh "${R2_PCA_PREFIX}".*

## Project round-1-passing AoU samples onto the round 2 PCs

Restricts the already biallelic-filtered ACAF pgen to (a) samples in `ROUND1_KEEP_PATH` and (b) variants in both `r2_pruned.prune.in` and `agreeing_snps.ids` -- reuses round 1's biallelic/agreement work rather than re-deriving it. AVG scoring (no `sum` modifier), matching round 1's convention -- both sides of every later comparison are `--score` output scored the same way, so they're on the same scale regardless of which convention that is.

In [ ]:
%%bash -s "$ACAF_BIALLELIC_PREFIX" "$AGREEING_SNPS_PATH" "$ROUND1_KEEP_PATH" "$R2_PRUNE_PREFIX" "$R2_PCA_PREFIX" "$R2_PROJECT_PREFIX" "$OUT_DIR"
set -e
ACAF_BIALLELIC_PREFIX=$1
AGREEING_SNPS_PATH=$2
ROUND1_KEEP_PATH=$3
R2_PRUNE_PREFIX=$4
R2_PCA_PREFIX=$5
R2_PROJECT_PREFIX=$6
OUT_DIR=$7

cd "$OUT_DIR"

# variants usable this round: round 2's pruned set, intersected with what's
# already known to agree (ID+REF+ALT) between ACAF and the reference
sort "${R2_PRUNE_PREFIX}.prune.in" > r2_pruned.sorted
sort "$AGREEING_SNPS_PATH" > agreeing.sorted
comm -12 r2_pruned.sorted agreeing.sorted > r2_usable_snps.ids
wc -l r2_usable_snps.ids

WEIGHTS="${R2_PCA_PREFIX}.eigenvec.allele"
HEADER=$(head -1 "$WEIGHTS")
ID_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'ID' | cut -d: -f1)
A1_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'A1' | cut -d: -f1)
PC1_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'PC1' | cut -d: -f1)
PC_LAST_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'PC20' | cut -d: -f1)

plink2 \
  --pfile "$ACAF_BIALLELIC_PREFIX" \
  --keep "$ROUND1_KEEP_PATH" \
  --extract r2_usable_snps.ids \
  --nonfounders \
  --read-freq "${R2_PCA_PREFIX}.acount" \
  --score "$WEIGHTS" "$ID_COL" "$A1_COL" header-read no-mean-imputation variance-standardize \
  --score-col-nums "${PC1_COL}-${PC_LAST_COL}" \
  --out "$R2_PROJECT_PREFIX"

head "${R2_PROJECT_PREFIX}.sscore"

## Sanity check: reproject the reference onto its own round 2 PCs

Same check as round 1 -- should closely recover the round 2 `--pca` eigenvectors (up to arbitrary per-PC sign). Restricted to `r2_usable_snps.ids` so it's on the identical variant set the AoU projection above used.

In [ ]:
%%bash -s "$OUT_PREFIX" "$r2_ref_keep_path" "$R2_PCA_PREFIX" "$R2_REPROJECT_PREFIX" "$OUT_DIR"
set -e
OUT_PREFIX=$1
REF_KEEP=$2
R2_PCA_PREFIX=$3
R2_REPROJECT_PREFIX=$4
OUT_DIR=$5

cd "$OUT_DIR"

WEIGHTS="${R2_PCA_PREFIX}.eigenvec.allele"
HEADER=$(head -1 "$WEIGHTS")
ID_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'ID' | cut -d: -f1)
A1_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'A1' | cut -d: -f1)
PC1_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'PC1' | cut -d: -f1)
PC_LAST_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'PC20' | cut -d: -f1)

plink2 \
  --bfile "$OUT_PREFIX" \
  --keep "$REF_KEEP" \
  --extract r2_usable_snps.ids \
  --nonfounders \
  --read-freq "${R2_PCA_PREFIX}.acount" \
  --score "$WEIGHTS" "$ID_COL" "$A1_COL" header-read no-mean-imputation variance-standardize \
  --score-col-nums "${PC1_COL}-${PC_LAST_COL}" \
  --out "$R2_REPROJECT_PREFIX"

head "${R2_REPROJECT_PREFIX}.sscore"

In [ ]:
direct = pd.read_csv(f"{R2_PCA_PREFIX}.eigenvec", sep=r"\s+")
reproj = pd.read_csv(f"{R2_REPROJECT_PREFIX}.sscore", sep=r"\s+")

direct_id_col = "#IID" if "#IID" in direct.columns else "IID"
merged = direct.merge(reproj, left_on=direct_id_col, right_on="IID", suffixes=("_direct", "_reproj"))

score_cols = [c for c in reproj.columns if c.startswith("PC") and c.endswith("_AVG")]
for i, pc in enumerate(range(1, 21)):
    direct_col = f"PC{pc}"
    score_col = score_cols[i] if i < len(score_cols) else None
    if direct_col in merged.columns and score_col in merged.columns:
        corr = merged[direct_col].corr(merged[score_col])
        print(f"PC{pc}: r = {corr:.4f}")

`round2_filter.ipynb` picks up from here: refits the Mahalanobis ellipsoid on `R2_PCA_PREFIX`'s reprojected reference scores, scores `R2_PROJECT_PREFIX`'s AoU output against it, and writes the final keep-list.